In [15]:
from pathlib import Path
import urllib.request

import torch
from transformers.models import deberta_v2


def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

data = download_shakespeare_text()

In [5]:
data[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [6]:
vocab = sorted(set(data.lower()))

In [9]:
vocab

['\n',
 ' ',
 '!',
 '$',
 '&',
 "'",
 ',',
 '-',
 '.',
 '3',
 ':',
 ';',
 '?',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [10]:
char_to_id = {char: ind for ind, char in enumerate(vocab)}
id_to_char = {ind: char for ind, char in enumerate(vocab)}

In [20]:
def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [21]:
decode_text(encode_text(data[:50]))

'first citizen:\nbefore we proceed any further, hear'

In [22]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, ind):
        end = ind + self.window_length
        window = self.encoded_text[ind: end]
        target = self.encoded_text[ind + 1: end + 1]
        return window, target

In [23]:
len(data)

1115394

In [25]:
window_length = 50
batch_size = 512
vocab_size = len(vocab)

train_set = CharDataset(data[:1_000_000], window_length)
valid_set = CharDataset(data[1_000_000:1_060_000], window_length)
test_set = CharDataset(data[1_060_000:], window_length)

train_loader = DataLoader(train_set, batch_size=batch_size, num_workers=4, pin_memory=True, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, num_workers=4, pin_memory=True)

In [26]:
import torch.nn as nn

class ShakespearModel(nn.Module):
    def __init__(self, vocab_size, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=256)
        self.gru = nn.GRU(input_size= self.embedding.embedding_dim,
                          hidden_size=128, num_layers=3, batch_first=True, dropout=dropout)
        self.output = nn.Linear(self.gru.hidden_size, vocab_size)

    def forward(self, X):
        X = self.embedding(X)
        outputs, hidden_states = self.gru(X)
        output = self.output(outputs)
        return output.permute(0, 2, 1)

In [34]:
from utils.utils import train
import torchmetrics

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ShakespearModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss().to(device)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size).to(device)

train(model, optimizer, criterion, train_loader, valid_loader, metric, 20, 5)

Epoch: 1/20, Loss: 1.6299, Val Score: 0.5381
Epoch: 2/20, Loss: 1.4157, Val Score: 0.5497
Epoch: 3/20, Loss: 1.3830, Val Score: 0.5545
Epoch: 4/20, Loss: 1.3661, Val Score: 0.5563
Epoch: 5/20, Loss: 1.3553, Val Score: 0.5566
Epoch: 6/20, Loss: 1.3475, Val Score: 0.5578
Epoch: 7/20, Loss: 1.3417, Val Score: 0.5575
Epoch: 8/20, Loss: 1.3371, Val Score: 0.5583
Epoch: 9/20, Loss: 1.3331, Val Score: 0.5574
Epoch: 10/20, Loss: 1.3298, Val Score: 0.5584
Epoch: 11/20, Loss: 1.3270, Val Score: 0.5579
Epoch: 12/20, Loss: 1.3244, Val Score: 0.5588
Epoch: 13/20, Loss: 1.3222, Val Score: 0.5585
Epoch: 14/20, Loss: 1.3202, Val Score: 0.5592


KeyboardInterrupt: 

In [35]:
def next_char(model, text, temperature=1.0, top_k=5):
    encoded = encode_text(text).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(encoded)
        last_logits = logits[0, :, -1]
        top_values, top_indices = torch.topk(last_logits, top_k)
        probas = torch.softmax(top_values / temperature, dim=-1)
        sampled_idx = torch.multinomial(probas, num_samples=1).item()
        predicted_id = top_indices[sampled_idx].item()
    return id_to_char[predicted_id]

In [40]:
def extend_text(model, text, n_chars=100, temperature=1.0, top_k=5):
    for _ in range(n_chars):
        text += next_char(model, text, temperature, top_k)
    return text

In [41]:
print(extend_text(model, "to be or not to b", temperature=0.2, top_k=5))
print("-"*50)
print(extend_text(model, "to be or not to b", temperature=1.0, top_k=5))
print("-"*50)
print(extend_text(model, "to be or not to b", temperature=5.0, top_k=5))
print("-"*50)

to be or not to be so.

claudio:
no, my lord, the senate is the sea,
and so much in the sea that the season should be
--------------------------------------------------
to be or not to be,
but in the weeling tongue that himself at these tells,
that tranio is not the death to the duke i
--------------------------------------------------
to be or not to be
thereir,
welrish,, at mayton trouples. an end;
fors iffunchion, whilsw hus all
i worthy,
stay:--wh
--------------------------------------------------
